# Fitbit — Data Exploration
Downloads the Takeout zip from Google Drive and runs the extractor logic.

In [ ]:
import io
import sys
import tempfile
from pathlib import Path

import gdown
import polars as pl
from dotenv import dotenv_values

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

env = dotenv_values(ROOT / '.env')
DRIVE_URL = env['DRIVE_SHARE_URL']
print('Drive URL:', DRIVE_URL)

In [ ]:
# Download fitbit subfolder from Drive
tmp = tempfile.mkdtemp()
gdown.download_folder(url=DRIVE_URL, output=tmp, use_cookies=False, quiet=False)

fitbit_dir = Path(tmp) / 'fitbit'
zip_files = list(fitbit_dir.glob('*.zip'))
print(f'Found {len(zip_files)} zip(s):', [f.name for f in zip_files])

In [ ]:
# Run extractor logic directly
from pipeline.extractors.fitbit import _extract, METRICS
from pipeline.r2 import R2Client

# Read zips locally (bypass R2)
import zipfile, json, re
from datetime import date

_FILE_RE = re.compile(
    r'Fitbit/Global Export Data/(calories|sleep|steps|exercise)-(\d{4}-\d{2}-\d{2})\.json$',
    re.IGNORECASE,
)

from pipeline.extractors.fitbit import _aggregate_day, _to_df

rows = {m: [] for m in METRICS}
for zp in zip_files:
    with zipfile.ZipFile(zp) as zf:
        for name in zf.namelist():
            m = _FILE_RE.search(name)
            if not m:
                continue
            metric, day_str = m.group(1).lower(), m.group(2)
            entries = json.loads(zf.read(name))
            row = _aggregate_day(metric, day_str, entries)
            if row:
                rows[metric].append(row)

frames = {metric: _to_df(r) for metric, r in rows.items() if r}
for metric, df in frames.items():
    print(f'{metric}: {len(df)} days, {df["date"].min()} → {df["date"].max()}')

In [ ]:
# Calories
frames['calories']

In [ ]:
# Steps
frames['steps']

In [ ]:
# Sleep
frames['sleep']

In [ ]:
# Exercise
frames['exercise']